# Run ExoMDN — LHS 6050 b

Loads the `mass_radius_Teq` model, runs an interior-structure inference from the
planet's mass, radius and equilibrium temperature, and saves the posterior samples
to `lhs6050b_samples.parquet`. Run this notebook first, then `plot_ridge_lhs6050b.ipynb`.

Place both notebooks anywhere inside your ExoMDN clone (e.g. next to `introduction.ipynb`).

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / "exomdn" / "mdn_model.py").is_file():
            return parent
    raise FileNotFoundError("Could not find ExoMDN repo root (no exomdn/mdn_model.py upward)")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from exomdn.mdn_model import Model

## Planet inputs

The only cell to edit. Values for LHS 6050 b; replace them to run a different planet.

In [ ]:
planet_name = "LHS 6050 b"

planet_mass = 5.2       # Earth masses
planet_mass_err = 2.2

planet_radius = 2.2     # Earth radii
planet_radius_err = 0.2

planet_teq = 188.0      # Kelvin
planet_teq_err = 6.0

## Load the model

In [ ]:
model = Model(REPO_ROOT / "models" / "mass_radius_Teq")

## Run the inference

`samples=(5000, 10)` draws 5000 Monte-Carlo inputs from the mass/radius/T_eq error bars
and 10 interior structures per input (~50 000 posterior samples). Inputs falling outside
the model's physical limits are dropped automatically and reported below.

In [ ]:
samples, components, used_inputs = model.predict_with_error(
    x=[planet_mass, planet_radius, planet_teq],
    errors=[planet_mass_err, planet_radius_err, planet_teq_err],
    samples=(5000, 10),
    enforce_teq_training_limits=True,
)

## Save the samples

In [ ]:
SAMPLES_PATH = Path.cwd() / "lhs6050b_samples.parquet"
samples.to_parquet(SAMPLES_PATH)
print(f"saved {len(samples):,} posterior samples \u2192 {SAMPLES_PATH.name}")